# **TUTORIAL** Study objective A (deterministic)

AESA Phase A: estimating the environmental burden.\
Compute **deterministic** IO-LCA outputs from processed MRIO tables.

# Before starting...

### Set workspace

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")

# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

### Run prerequisites

This tutorial assumes that the workspace has already been set, with all prerequisites available (downloads and processing).\
If this is not your case, it is recommended to first go through the core prerequisite notebooks: [tutorials/core_prerequisites/0_set_workspace.ipynb](../../core_prerequisites/0_set_workspace.ipynb), [tutorials/core_prerequisites/1_download_data.ipynb](../../core_prerequisites/1_download_data.ipynb), and [tutorials/core_prerequisites/2_process_data.ipynb](../../core_prerequisites/2_process_data.ipynb).

### Functional units and selectors

The example in this tutorial uses the **functional unit** `L2.c.b` with the following **selectors**: one producing sector `s_p="Paper"` and one consuming region `r_c="FR"`.

Use [tutorials/study_objectives/1_functional_units_and_allocation_methods.md](../1_functional_units_and_allocation_methods.md) to choose the FU and required selectors. This IO-LCA notebook uses only the FU and selector parts of that guide. Use <a href="../../methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf">methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</a> for detailed FU definitions and mathematical expressions.

In [ ]:
project_name_ = "SO_A_demo_paper_fr"

source_ = "exiobase_3102_ixi"
fu_code_ = "L2.c.b"
study_period_ = range(2010, 2021)
s_p_ = ["Paper"]
r_c_ = ["FR"]
lcia_method_ = "pb_lcia"

# Basic features of the deterministic function

### In a nutshell

The function takes necessary **inputs** such as `project_name`, `source`, `years`, `lcia_method`, `fu_code` and the relevant selectors (here `s_p` and `r_c`) for the selected functional unit.

IO-LCA currently requires EXIOBASE processed with an LCIA method. Characterization matrices for EXIOBASE are available for PB-LCIA and GWP100.

The deterministic **output** folder contains:
- IO-LCA result tables.
- **Figures** are rendered by default (`figures=True`). Use `figures=False` to skip them; `figure_format` controls the file format and resolution.
- log files

Basic features also involve:
- **MRIO aggregation and disaggregation** of sectors and/or regions: use the `agg_sec`, `agg_reg`, and `agg_version` parameters. The MRIO aggregation folder includes packaged `agg_reg_eu27.csv` and `agg_reg_world.csv` examples; inspect them in `data_raw/mrio/<source>/aggregation/` and follow `README_aggregation.txt` before writing a custom MRIO aggregation and disaggregation mapping CSV.
- **overwriting** of existing values: use the `refresh` parameter.

### Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>deterministic_io_lca(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>project_name</code></td><td>Required project name used to build<br>&#10;<code>&lt;repo&gt;/&lt;project_name&gt;</code>.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>source</code></td><td>MRIO source key (<code>&quot;exiobase_396_ixi&quot;</code>,<br>&#10;<code>&quot;exiobase_396_pxp&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;<code>&quot;exiobase_3102_pxp&quot;</code>). <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> currently only supports<br>&#10;EXIOBASE for LCIA characterization.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>agg_reg</code></td><td>If <code>True</code>, reclassify MRIO regions with the <code>agg_reg_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native regions into one target label, or disaggregate one native region across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source regions.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>agg_sec</code></td><td>If <code>True</code>, reclassify MRIO sectors with the <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping. The mapping can keep native labels, aggregate several native sectors into one target label, or disaggregate one native sector across several target labels when a <code>weight</code> column is provided.<br>&#10;<strong>Default</strong> <code>False</code> keeps native source sectors.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>agg_version</code></td><td>Name token used to resolve the matching <code>agg_reg_&lt;agg_version&gt;.csv</code> and/or <code>agg_sec_&lt;agg_version&gt;.csv</code> MRIO aggregation and disaggregation mapping files in <code>data_raw/mrio/&lt;source&gt;/aggregation</code>. Required when <code>agg_reg</code> or <code>agg_sec</code> is True.<br>&#10;<strong>Defaults to</strong> an empty string for native source classification. Use the same token in downstream calls that should reuse the processed classification. If a mapping file has a <code>weight</code> column, weights must sum to <code>1</code> for each original label.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>years</code></td><td>Studied years. Accepts a single year, list, or range. If<br>&#10;<strong>omitted</strong>, all available MRIO<br>&#10;years for the selected source and <code>agg_version</code> are used.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>lcia_method</code></td><td>Required LCIA method(s) selected for IO-LCA results<br>&#10;(for example <code>&quot;pb_lcia&quot;</code> or <code>[&quot;pb_lcia&quot;, &quot;gwp100_lcia&quot;]</code>).<br>&#10;The method(s) must have been processed for the same MRIO source<br>&#10;with <code>process_mrio(...)</code>. <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> currently supports IO-LCA only<br>&#10;for EXIOBASE sources. To add a custom LCIA method with which run<br>&#10;<code>process_mrio(...)</code>, follow<br>&#10;<code>README_add_custom_lcia_characterization_matrices.txt</code> in<br>&#10;<code>data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/</code><br>&#10;and pass the custom method file stem here.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>fu_code</code></td><td>Required functional unit code (for example <code>&quot;L1.a&quot;</code>,<br>&#10;<code>&quot;L2.c.b&quot;</code>). See<br>&#10;<code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;for all available functional unit codes and the system<br>&#10;boundaries each represents.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>s_p</code></td><td>Producing sector filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid producing sectors. To identify valid sector<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/.../agg_sec_template.csv</code> file. For<br>&#10;EXIOBASE sector definitions, see<br>&#10;<code>data_raw/mrio/exiobase_3/sector_classification.xlsx</code>; EXIOBASE<br>&#10;ixi and pxp use different sector lists.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_p</code></td><td>Producing region filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid producing regions. To identify valid region<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code> file.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_c</code></td><td>Consuming region filter(s), single string or list. If this is a<br>&#10;required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the run<br>&#10;expands to all valid consuming regions. To identify valid region<br>&#10;names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code> file.</td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>r_f</code></td><td>Final demand region filter(s), single string or list. If this is<br>&#10;a required axis for <code>fu_code</code> and the argument is <strong>omitted</strong>, the<br>&#10;run expands to all valid final demand regions. To identify valid<br>&#10;region names, see the first column of the relevant<br>&#10;<code>data_raw/mrio/.../aggregation/agg_reg_template.csv</code> file.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>upstream_analysis</code></td><td>Whether upstream diagnostic outputs are written.<br>&#10;These outputs are for user audit only and are not used by any<br>&#10;downstream package function. <strong>Default is</strong> <code>False</code>.<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>False</code>: write only main IO-LCA result tables.<br>&#10;&bull;&nbsp;<code>True</code>: also write origin tables attributing the footprint to<br>&#10;  producer sector-region pairs where impacts occur, and when<br>&#10;  supported by the FU, stage tables showing each upstream supply<br>&#10;  chain step with direct, embedded, and total impacts.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>upstream_stages</code></td><td>Number of upstream supply chain steps written when<br>&#10;<code>upstream_analysis=True</code>. <strong>Default</strong> <code>3</code> writes <code>n</code> to<br>&#10;<code>n-3</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>group_indices</code></td><td>Whether multiple selected region or sector filter values are kept as separate result rows or summed into one result row after the function calculation has been performed.<br>&#10;&bull;&nbsp;<code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;&bull;&nbsp;<code>True</code>: sum selected values into one result row.<br>&#10;The function refuses to run when <code>group_indices=True</code> is used with <code>L2.a.b</code>, <code>L2.b.b</code>, or <code>L2.c.b</code> because summing output rows for CBA total demand boundaries can double count. For these functional units, change the upstream MRIO aggregation and disaggregation scope with <code>agg_reg</code>, <code>agg_sec</code>, and <code>agg_version</code> before running the study.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Persisted output file format: <code>&quot;csv&quot;</code> (<strong>default</strong>),<br>&#10;<code>&quot;pickle&quot;</code>, or <code>&quot;parquet&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull;&nbsp;<code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull;&nbsp;<code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, clear and recompute the resolved deterministic IO-LCA source and version output scope under <code>&lt;project&gt;/A_lca/io_lca</code>. It is not limited to one LCIA method inside that output scope. Processed MRIO inputs, processed population and GDP, raw downloads, and downstream ASR outputs are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>

### Deterministic IO-LCA with defaults where omitted

In [ ]:
from pyaesa import deterministic_io_lca

deterministic_io_lca(
    project_name=project_name_,
    source=source_,
    years=study_period_,
    lcia_method=lcia_method_,
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
)

### External deterministic LCA results

If you want to use **externally provided LCA results** instead of <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> IO-LCA results computed with EXIOBASE for the ASR numerator, use [tutorials/optional_workflows/external_asocc_lca_input_staging.ipynb](../../optional_workflows/external_asocc_lca_input_staging.ipynb).

# What to do next

**Beginners**

If you are a new user in the process of discovering <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span>, it is recommend that you first discover all study objectives with the **basic features** available.
Have a look at the other notebooks available at [tutorials/study_objectives/0_study_objectives.md](../0_study_objectives.md)

**Experts**

If you are already familiar with <span style="color:#366e9c"><strong>py</strong></span><span style="color:#c83737"><strong>aesa</strong></span> and if you want to discover **advanced features** available, check out examples in the next section of this tutorial !

# Advanced features

Advanced features currently available include:
- upstream analysis
- custom LCIA method
- indices aggregation

### Upstream analysis

Optional upstream origin
and supply chain stage outputs are written only when `upstream_analysis=True`.

In [ ]:
deterministic_io_lca(
    project_name=project_name_,
    source=source_,
    years=study_period_,
    lcia_method=lcia_method_,
    fu_code=fu_code_,
    s_p=s_p_,
    r_c=r_c_,
    upstream_analysis=True,
)

### Indices aggregation

`group_indices=True` reports multiple selected region or sector indices as one
summed result row after the selected IO-LCA scope is computed. The default
`False` keeps selected indices as separate rows.

The function refuses to run when `group_indices=True` is used with `L2.a.b`,
`L2.b.b`, or `L2.c.b`, because summing output rows for CBA total demand boundaries
can double count. For those functional units, define aggregated regions or sectors
during MRIO processing with `agg_reg`, `agg_sec`, and `agg_version`.

The basic notebook example uses `L2.c.b`, so this aggregation example switches
to `L2.c.a`.

In [ ]:
deterministic_io_lca(
    project_name=f"{project_name_}_group_indices",
    source=source_,
    years=study_period_,
    lcia_method=lcia_method_,
    fu_code="L2.c.a",
    s_p=["Paper"],
    r_c=["FR", "DE"],
    group_indices=True,
)

### Custom LCIA method

To
add a custom LCIA method, follow
`data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/README_add_custom_lcia_characterization_matrices.txt`
and run `process_mrio(...)` with the custom method file stem.